In [1]:
!pip install -U bitsandbytes>=0.46.1

In [2]:
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM 
from peft import LoraConfig, get_peft_model 
from datasets import load_dataset 
import torch 

In [3]:
configuration = BitsAndBytesConfig(
    load_in_4bit = True, 
    bnb_4bit_quant_type="nf4", 
    bnb_4bit_compute_dtype= torch.float16,
    bnb_4bit_use_double_quant=True 
)

In [4]:
model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2", quantization_config = configuration, device_map="auto", torch_dtype = torch.float16)
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [5]:
data_set = load_dataset("tatsu-lab/alpaca")
print(data_set['train'][0])
print(data_set)

{'instruction': 'Give three tips for staying healthy.', 'input': '', 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.', 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}
DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 52002
    })
})


In [6]:
def tokenizing(text):
    
    x = tokenizer(text, truncation = True, max_length = 512)
    x["labels"] = x["input_ids"] 
    
    return x 
    

In [7]:
tokenized = data_set.map(tokenizing, input_columns = ["text"], remove_columns = data_set["train"].column_names) 


In [8]:
configure_lora = LoraConfig(
    r = 8,
    lora_alpha = 16, 
    target_modules = ["q_proj", "v_proj"],
    lora_dropout = 0.1, 
    task_type = "CAUSAL_LM"
    
)

In [9]:
!pip install trl 


In [14]:
from trl import SFTTrainer 
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = ".mistral-lora-alpaca",
    num_train_epochs = 1, 
    per_device_train_batch_size = 2, 
    gradient_accumulation_steps = 4, 
    learning_rate = 2e-4, 
    bf16 = False,
    logging_steps = 50, 
    save_steps = 100,
    fp16 = False,
    max_steps = 200
    
    
)

In [15]:
trainer = SFTTrainer(
    model = model ,
    args = training_args, 
    train_dataset= tokenized["train"],
    processing_class = tokenizer, 
    peft_config = configure_lora 
   
    
    
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [16]:
trainer.train() 

Step,Training Loss
50,1.103818
100,0.977155
150,0.980933
200,0.982541


TrainOutput(global_step=200, training_loss=1.0111119079589843, metrics={'train_runtime': 648.5763, 'train_samples_per_second': 2.467, 'train_steps_per_second': 0.308, 'total_flos': 1.0566781259317248e+16, 'train_loss': 1.0111119079589843})

In [19]:
inputs = tokenizer("### Instruction:\nExplain why gradient descent can get stuck in a local minimum and not find the global minimum.\n\n### Response:", return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Explain why gradient descent can get stuck in a local minimum and not find the global minimum.

### Response:
Gradient descent is an optimization algorithm that uses the gradient of a function to find the minimum of that function. However, it can get stuck in a local minimum and not find the global minimum if the gradient of the function is not pointing towards the


In [ ]:
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470


In [20]:
model.save_pretrained("mistral-lora-adapters")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]